# Task 6 — RAG-Powered Q&A (FAISS)

A retrieval-augmented question answering notebook. It indexes a small domain knowledge base, retrieves the passages most relevant to a question with a **FAISS** vector index, and generates an answer that is grounded in those passages and cites its sources.

**Pipeline:** documents → chunk → embed (`all-MiniLM-L6-v2`) → FAISS index → retrieve top-k → LLM answer grounded in the retrieved context.

The embedding and retrieval stack reuses Task 5; the generation step reuses the OpenAI model from Task 3. Without an `OPENAI_API_KEY` the notebook still runs and returns the retrieved context, so you can watch retrieval work before wiring up an LLM.

## 1. Setup

Imports, model names, and where the index is persisted.

In [ ]:
import json
import os
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TOP_K = 3

INDEX_DIR = Path("rag_index")
FAISS_PATH = INDEX_DIR / "faiss.index"
CHUNKS_PATH = INDEX_DIR / "chunks.json"

## 2. Domain knowledge base

A small set of documents for a fictional product, **Lumen**. Because the language model has never seen this product, correct answers can only come from retrieval — which is exactly what RAG is for. Swap this list for your own documents to use the notebook for real.

In [ ]:
DOCUMENTS = [
    {"title": "Plans & Pricing", "text": "Lumen comes in four plans. Free is $0 and covers personal use. Pro is $8 per month. Team is $12 per user per month. Enterprise uses custom pricing negotiated with sales. Choosing annual billing instead of monthly saves 20 percent on Pro and Team."},
    {"title": "Free Plan Limits", "text": "The Free plan includes 1 GB of storage, up to 3 notebooks, and full-text search. Sync is limited to two devices and version history is not included. Free workspaces cannot invite collaborators."},
    {"title": "Pro Features", "text": "Pro unlocks unlimited notebooks, 50 GB of storage, offline editing, and 30 days of version history. Pro also enables semantic search, which finds notes by meaning rather than exact keywords. File uploads on Pro are capped at 100 MB each."},
    {"title": "Team Plan", "text": "Team adds shared workspaces, a central admin console, and single sign-on through SAML. It includes 90 days of version history, audit logs, and 250 MB file uploads. Members share a pooled 100 GB of storage."},
    {"title": "Enterprise Plan", "text": "Enterprise offers everything in Team plus SCIM user provisioning, a 99.9 percent uptime SLA, choice of data residency, and a dedicated customer success manager. Onboarding and security reviews are handled by the Lumen solutions team."},
    {"title": "Security", "text": "Lumen encrypts data at rest with AES-256 and data in transit with TLS 1.3. Notes placed in a Vault support optional end-to-end encryption, meaning Lumen servers cannot read their contents. Lumen maintains a SOC 2 Type II report available under NDA."},
    {"title": "Data Residency", "text": "Workspace data can be stored in the United States, the European Union (Frankfurt), or Asia Pacific (Singapore). The region is selected when the workspace is created and cannot be changed afterward. Region choice is available on Enterprise plans."},
    {"title": "Sync & Offline", "text": "Edits sync in real time across signed-in devices. Offline editing is available on Pro and above; changes made offline are merged when the device reconnects. Conflicts are resolved last-writer-wins, and the overwritten version is kept in history so nothing is lost."},
    {"title": "Search", "text": "Every plan includes full-text search across note titles and bodies. Semantic search, which ranks results by meaning, is available on Pro and above. Search does not cover the contents of attached files."},
    {"title": "Sharing & Permissions", "text": "Notes and notebooks can be shared with view, comment, or edit permissions. Public share links can be created with an optional expiry date and password. Collaboration features require a Team or Enterprise plan."},
    {"title": "Integrations & API", "text": "Lumen integrates with Slack, Google Drive, and Zapier, and can import from Notion. The REST API authenticates with personal access tokens and is rate limited to 600 requests per minute per token. Webhooks are available on Team and Enterprise."},
    {"title": "Billing & Refunds", "text": "Plans can be paid monthly or annually and cancelled at any time. Annual plans are eligible for a full refund within 14 days of purchase. After cancellation, you can export all of your data as Markdown or JSON for up to 30 days before it is deleted."},
    {"title": "Support", "text": "All plans include email support. Live chat is available on Pro and above, with a target first response within 8 hours. Enterprise customers receive phone support and a 1-hour response target for urgent issues."},
]

len(DOCUMENTS)

## 3. Chunk the documents

Split each document into overlapping word windows so long passages can be retrieved precisely. Every chunk keeps its source title so answers can cite where they came from. Short documents stay as a single chunk.

In [ ]:
def chunk_documents(documents, chunk_size=120, overlap=30):
    chunks = []
    for doc in documents:
        words = doc["text"].split()
        start, part = 0, 0
        while start < len(words):
            window = words[start:start + chunk_size]
            chunks.append({
                "source": doc["title"],
                "chunk_id": f"{doc['title']}#{part}",
                "text": " ".join(window),
            })
            if start + chunk_size >= len(words):
                break
            start += chunk_size - overlap
            part += 1
    return chunks


chunks = chunk_documents(DOCUMENTS)
len(chunks)

## 4. Embed and build the FAISS index

Encode every chunk into a normalized vector and load them into a FAISS inner-product index. Because the vectors are normalized, inner product is equivalent to cosine similarity. The index and its chunk metadata are written to `rag_index/` on the first run and reused on later runs.

In [ ]:
_model = None


def get_model():
    global _model
    if _model is None:
        _model = SentenceTransformer(EMBED_MODEL)
    return _model


def embed(texts):
    vectors = get_model().encode(list(texts), convert_to_numpy=True, normalize_embeddings=True)
    return vectors.astype("float32")


def build_index(chunks):
    embeddings = embed([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(FAISS_PATH))
    CHUNKS_PATH.write_text(json.dumps(chunks, indent=2), encoding="utf-8")
    return index, chunks


def load_or_build_index(chunks):
    if FAISS_PATH.exists() and CHUNKS_PATH.exists():
        index = faiss.read_index(str(FAISS_PATH))
        return index, json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
    return build_index(chunks)


index, chunks = load_or_build_index(chunks)
print(f"Indexed {index.ntotal} chunks from {len(DOCUMENTS)} documents.")

## 5. Retrieve relevant chunks

Embed the query and ask FAISS for the closest chunks. Each result carries its source and a similarity score.

In [ ]:
def retrieve(query, k=TOP_K):
    scores, idx = index.search(embed([query]), k)
    return [
        {**chunks[i], "score": float(s)}
        for s, i in zip(scores[0], idx[0]) if i != -1
    ]


for hit in retrieve("How much does the team plan cost?"):
    print(f"[{hit['score']:.3f}] {hit['source']}: {hit['text'][:90]}...")

## 6. Generate a grounded answer

Feed the retrieved chunks to the LLM as context and instruct it to answer only from that context and cite its sources. If `OPENAI_API_KEY` is not set, the retrieved context is returned directly so the retrieval half still works without a key.

In [ ]:
SYSTEM_PROMPT = (
    "You are a support assistant for the Lumen product. "
    "Answer the question using only the context provided. "
    "If the context does not contain the answer, say you do not have that information. "
    "Cite the sources you used in square brackets, for example [Security]."
)


def format_context(hits):
    return "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)


def generate(question, hits):
    context = format_context(hits)
    if not os.getenv("OPENAI_API_KEY"):
        sources = ", ".join(sorted({h["source"] for h in hits}))
        return f"(No OPENAI_API_KEY set — showing retrieved context only)\n\n{context}\n\nSources: {sources}"
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_openai import ChatOpenAI

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ])
    chain = prompt | ChatOpenAI(model=LLM_MODEL, temperature=0) | StrOutputParser()
    return chain.invoke({"context": context, "question": question})


def answer(question, k=TOP_K):
    hits = retrieve(question, k)
    return generate(question, hits), hits

## 7. Ask domain questions

Each answer is grounded in the retrieved passages and lists the sources it drew from.

In [ ]:
QUESTIONS = [
    "How much does the Team plan cost?",
    "Can I use Lumen offline, and what happens to conflicting edits?",
    "How is my data encrypted?",
    "What is the API rate limit?",
    "Which regions can I store my data in?",
]

for q in QUESTIONS:
    response, hits = answer(q)
    print("Q:", q)
    print("A:", response)
    print("Sources:", ", ".join(sorted({h["source"] for h in hits})))
    print("-" * 80)

## 8. A question outside the knowledge base

Nothing in the corpus mentions video calls, so a grounded assistant should decline rather than invent an answer. The retrieved chunks still come back, but with low scores.

In [ ]:
response, hits = answer("Does Lumen offer live video calls?")
print(response)
print("\nRetrieved (low relevance):", [(h["source"], round(h["score"], 3)) for h in hits])

## Using your own data

1. Replace the `DOCUMENTS` list with your own `{"title": ..., "text": ...}` entries, or load them from files, a database, or an API.
2. Delete the `rag_index/` folder so the index rebuilds against the new content.
3. Re-run the notebook from top to bottom.

Tune `chunk_size` and `overlap` to suit your document length, and `TOP_K` for how much context each answer sees. For large collections, swap `IndexFlatIP` for an approximate FAISS index such as `IndexIVFFlat` — the `retrieve` and `answer` interface stays the same.